# Memory AI Lab — Évaluation ARI V3

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Protocole anti-surapprentissage

```
group_gold_tune.json  → 448 épisodes, 8954 msgs  (août 2023 → jan 2025)
                         ↑ GRID SEARCH ICI — tuner les paramètres

group_gold_test.json  → 193 épisodes, 2786 msgs  (jan 2025 → mars 2026)
                         ↑ SCORE FINAL — une seule fois, ne pas tuner dessus
```

**Données requises sur Google Drive (`memory_ai_data/`) :**
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
```

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull
else:
    !git clone {REPO} {CODE_DIR}
import sys
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')

In [ ]:
# ── CELLULE 3 : Drive + GPU ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/memory_ai_data'
for f in ['group_anon.txt', 'group_gold_tune.json', 'group_gold_test.json']:
    assert os.path.exists(f'{DATA_DIR}/{f}'), f'Manquant : {f}'
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ Drive OK | {device}', torch.cuda.get_device_name(0) if device=='cuda' else '')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings complets (GPU + cache) ─────────────────
# On embède TOUS les messages une fois — tune et test y piochent ensuite
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings.npy'
all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings.npy'
    print('[2/2] Embeddings chargés depuis cache Drive')
else:
    print(f'[2/2] Calcul embeddings sur {device} ...')
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
    all_embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                                  device=device, convert_to_numpy=True).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')
print(f'      Shape : {all_embeddings.shape}')

In [ ]:
# ── CELLULE 5 : Charger tune et test séparément ────────────────────────────
import json

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts, y_true_tune, tune_eps, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts, y_true_test, test_eps, test_meta = load_split(f'{DATA_DIR}/group_gold_test.json')

# Tranches d'embeddings correspondantes
n_tune = len(tune_arts)
emb_tune = all_embeddings[:n_tune]
emb_test = all_embeddings[n_tune:n_tune + len(test_arts)]

# Reconstruire les artifacts Python depuis all_artifacts
arts_tune = all_artifacts[:n_tune]
arts_test = all_artifacts[n_tune:n_tune + len(test_arts)]

print(f'✓ Tune : {len(tune_eps)} épisodes · {n_tune} msgs · {tune_meta["period"]}')
print(f'✓ Test : {len(test_eps)} épisodes · {len(test_arts)} msgs · {test_meta["period"]}')
print()
print('⚠️  Le score TEST ne doit être calculé qu\'une seule fois, avec les paramètres finaux.')

In [ ]:
# ── CELLULE 6 : Fonctions helper ──────────────────────────────────────────
from episode_algorithm import EpisodeSegmenter
from episode_splitter import EpisodeSplitter, SplitConfig
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SPLITTER = EpisodeSplitter(SplitConfig(
    min_cohesion=0.65, min_size_to_split=8,
    max_span_hours=168.0, max_splits=6,
    min_sub_size=3, silhouette_threshold=0.10,
))

def run_eval(artifacts, embeddings, y_true, seg_kwargs, split_name=''):
    seg = EpisodeSegmenter(**seg_kwargs)
    eps = seg.consolidate(seg.segment(artifacts, embeddings))
    eps = SPLITTER.split(eps, artifacts, embeddings)
    n = len(artifacts)
    y_pred = [None] * n
    for ep in eps:
        for idx in ep.artifact_indices:
            if idx < n: y_pred[idx] = ep.id
    pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
    yt, yp = zip(*pairs)
    ari = adjusted_rand_score(yt, yp)
    nmi = normalized_mutual_info_score(yt, yp)
    n_gold = len(set(t for t in y_true if t is not None))
    label = f'[{split_name}] ' if split_name else ''
    print(f'{label}ARI={ari:+.4f}  NMI={nmi:.4f}  gold={n_gold}  pred={len(eps)}')
    return ari, nmi, len(eps)

print('✓ Helpers prêts')

In [ ]:
# ── CELLULE 7 : Grid search sur TUNE uniquement ────────────────────────────
# ⚠️  NE PAS regarder le score TEST ici — uniquement TUNE
import itertools, pandas as pd

results = []
for attach, hb, time_thr in itertools.product(
    [0.30, 0.35, 0.40, 0.45, 0.50],   # attach_threshold
    [0, 1440, 2880, 4320],             # hard_break_minutes (0=désactivé)
    [120, 240, 360],                   # time_threshold_minutes
):
    seg_kwargs = dict(
        time_threshold_minutes=time_thr,
        attach_threshold=attach,
        alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
        dormancy_minutes=1440, ema_alpha=0.80, active_penalty_hours=24.0,
        hard_break_minutes=hb, allow_reactivation=True,
    )
    ari, nmi, n_ep = run_eval(arts_tune, emb_tune, y_true_tune, seg_kwargs, 'TUNE')
    results.append({'attach': attach, 'hard_break': hb, 'time_thr': time_thr,
                    'n_ep': n_ep, 'ari': ari, 'nmi': nmi})

df = pd.DataFrame(results).sort_values('ari', ascending=False)
print('\n── Top 10 (TUNE) ──')
print(df.head(10).to_string(index=False))

best = df.iloc[0].to_dict()
print(f'\n✓ Meilleurs params : attach={best["attach"]}  hard_break={best["hard_break"]}  time_thr={best["time_thr"]}')

In [ ]:
# ── CELLULE 8 : Score FINAL sur TEST (une seule fois) ─────────────────────
# Copier ici les meilleurs paramètres trouvés en cellule 7
# Exécuter UNE SEULE FOIS — c'est le score officiel

BEST_PARAMS = dict(
    time_threshold_minutes=120,   # ← remplacer avec le meilleur
    attach_threshold=0.30,        # ← remplacer avec le meilleur
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes=1440, ema_alpha=0.80, active_penalty_hours=24.0,
    hard_break_minutes=720,       # ← remplacer avec le meilleur
    allow_reactivation=True,
)

print('Score TUNE (référence) :')
ari_tune, _, _ = run_eval(arts_tune, emb_tune, y_true_tune, BEST_PARAMS, 'TUNE')

print('\nScore TEST (officiel) :')
ari_test, nmi_test, n_ep_test = run_eval(arts_test, emb_test, y_true_test, BEST_PARAMS, 'TEST')

gap = ari_tune - ari_test
print(f'\nGap tune-test : {gap:+.4f}', '← OK' if gap < 0.05 else '← SURAPPRENTISSAGE')
print()
print(f"""
╔══════════════════════════════════════════════╗
║  RÉSULTAT OFFICIEL V3                        ║
║  ARI  (test)  : {ari_test:+.4f}              ║
║  NMI  (test)  : {nmi_test:.4f}               ║
║  Gold (test)  : {len(test_eps)} épisodes     ║
║  Pred (test)  : {n_ep_test} épisodes         ║
╚══════════════════════════════════════════════╝
""")